# Corporate Insolvency in India: The Intangible Asset Illusion & Forensic Modeling

## Summary
The Indian corporate landscape has witnessed spectacular financial collapses—from Kingfisher Airlines to the Amtek Auto group. Traditional financial screeners often miss these events because they rely on easily manipulated metrics like Net Income or top-line revenue. 

This project builds a predictive insolvency model (Random Forest) focused on forensic accounting indicators. Rather than just looking at negative profitability, we are testing specific failure signatures:
1. **The Intangible Asset Illusion:** Distressed companies frequently inflate their balance sheets using heavily impaired "intangible assets" or "goodwill" to secure leverage, masking an underlying negative net worth.
2. **The Working Capital Trap:** A severe divergence between reported EBITDA (paper profit) and Operating Cash Flow (actual liquidity), usually driven by uncollectible receivables.
3. **The Debt Avalanche:** Capital structures overly reliant on short-term borrowings to fund long-term, depreciating assets, leading to sudden liquidity crises when credit markets tighten.

## The Dataset
We are utilizing `insolvency_raw.csv`, a dataset containing ~400 Indian listed companies. Crucially, the "bankrupt" class (Target = 1) features a curated list of corporate insolvencies gathered through standard financial platforms (such as Yahoo Finance, Screener, and Investing.com). This data is fortified by rigorous manual vetting of company annual reports from the fiscal years immediately preceding their collapse, capturing their financial state at the exact moment the terminal distress became mathematically irreversible.

## Dataset Features

### Metadata Fields
- `ticker`: Stock ticker symbol (e.g., RELIANCE, TCS)
- `company_name`: Legal company name
- `sector`: Industry sector
- `industry`: Industry classification
- `fiscal_year`: Financial year of data
- `currency`: Currency of financial values (INR)
- `source_url`: Where the data was sourced from
- `extraction_status`: Data collection status
- `extraction_error`: Any errors during data extraction

### Financial Features (Key Indicators)
- `market_cap`: Market capitalization
- `total_debt`: Total debt outstanding
- `intangible_assets`: Goodwill and intangible assets
- `cash_and_equivalents`: Cash on hand
- `current_liabilities`: Short-term obligations
- `operating_cash_flow`: Cash generated from operations
- `ebitda`: Earnings before interest, tax, depreciation, amortization
- `interest_expense`: Interest paid on debt
- `net_income`: Profit/loss for the period
- `total_assets`: Total resources controlled by company

### Target Variable
- `target`: 1 (Insolvent/Bankrupt), 0 (Healthy/Solvent)

### Environment Setup

In [4]:
# Standard Data Stack
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# ML & Preprocessing
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve

# Configuration
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)
import warnings
warnings.filterwarnings('ignore')

print("Environment Initialized.")

Environment Initialized.


## Data Cleaning & Imputation

In [25]:
# Load data and check initial state
df_raw = pd.read_csv('insolvency_raw.csv', low_memory=False)

print(f"Raw shape: {df_raw.shape}")
display(df_raw.info())

# Check class imbalance
print("\nTarget Distribution:")
print(df_raw['target'].value_counts(normalize=True, dropna=False))

# Identify duplicate tickers
dupes = df_raw[df_raw.duplicated(subset=['ticker'], keep=False)]
print(f"\nFound {len(dupes)} rows with duplicate tickers.")
if len(dupes) > 0:
    display(dupes[['ticker', 'company_name', 'extraction_status']].sort_values('ticker').head())

Raw shape: (416, 21)
<class 'pandas.DataFrame'>
RangeIndex: 416 entries, 0 to 415
Data columns (total 21 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   ticker                416 non-null    str    
 1   company_name          416 non-null    str    
 2   sector                370 non-null    str    
 3   industry              370 non-null    str    
 4   business_summary      370 non-null    str    
 5   target                416 non-null    int64  
 6   market_cap            383 non-null    str    
 7   total_debt            381 non-null    str    
 8   intangible_assets     377 non-null    str    
 9   cash_and_equivalents  383 non-null    str    
 10  current_liabilities   383 non-null    str    
 11  operating_cash_flow   383 non-null    str    
 12  ebitda                381 non-null    str    
 13  interest_expense      382 non-null    str    
 14  net_income            382 non-null    str    
 15  total_assets 

None


Target Distribution:
target
0   0.67
1   0.33
Name: proportion, dtype: float64

Found 40 rows with duplicate tickers.


,ticker,company_name,extraction_status
0,ALOKINDS,ALOK INDUSTRIES LIMITED,yfinance
1,ALOKINDS,Alok Industries Ltd,success_manual
32,ANSALAPI,ANSAL PROP & INFRA LTD,Success_Manual
33,ANSALAPI,Ansal Properties & Infrastructure,yfinance
31,ARSSINFRA,ARSS Infrastructure Projects,yfinance


## Cleaning Execution

From this point onward, we apply transformations to fix the issues diagnosed in cells 6 to 10.

Cleaning goals:
- Preserve key identifier columns (`ticker`, `company_name`, `target`)
- Standardize text and URL formats
- Convert numeric fields from comma-formatted strings to numeric dtype
- Resolve duplicate tickers using data completeness
- Handle missing values in financial columns
- Validate dataset integrity before EDA

In [33]:
# Create a working copy (STRICT CLEANING: drop incomplete financial rows)
df = df_raw.copy()

financial_cols = ['market_cap', 'total_debt', 'intangible_assets', 'cash_and_equivalents',
                  'current_liabilities', 'operating_cash_flow', 'ebitda', 'interest_expense',
                  'net_income', 'total_assets']

# 1) Standardize text fields (preserve missing values)
for col in ['company_name', 'sector', 'industry', 'extraction_status']:
    df[col] = df[col].astype('string').str.strip()

df['sector'] = df['sector'].str.title()
df['industry'] = df['industry'].str.title()
df['extraction_status'] = df['extraction_status'].str.upper()

# URL normalization helper
import re

def normalize_url(url):
    if pd.isna(url):
        return np.nan
    s = str(url).strip()
    if s == '' or s.lower() in {'nan', 'none', 'null', 'n/a'}:
        return np.nan

    # Remove internal whitespace and protocol artifacts
    s = re.sub(r'\s+', '', s)
    s = s.replace('yahoo://', '')
    s = re.sub(r'^(https?://)+', 'https://', s)

    # Force https scheme if absent or http
    if s.startswith('http://'):
        s = 'https://' + s[len('http://'):]
    elif not s.startswith('https://'):
        s = 'https://' + s.lstrip('/')

    return s

df['source_url'] = df['source_url'].apply(normalize_url)

# 2) Numeric normalization (remove commas and coerce errors)
for col in financial_cols:
    df[col] = pd.to_numeric(df[col].astype(str).str.replace(',', ''), errors='coerce')

df['fiscal_year'] = pd.to_numeric(df['fiscal_year'], errors='coerce').astype('Int64')

# 3) Remove low-quality rows
rows_start = len(df)
metadata_only_mask = df['extraction_status'].fillna('').str.contains('METADATA_ONLY', na=False)
df = df[~metadata_only_mask].copy()
removed_metadata_only = int(metadata_only_mask.sum())

missing_ratio = df[financial_cols].isnull().sum(axis=1) / len(financial_cols)
high_missing_mask = missing_ratio > 0.5
df = df[~high_missing_mask].copy()
removed_high_missing = int(high_missing_mask.sum())

# 4) Resolve duplicates (prefer least missing financials, then latest fiscal year)
df['missing_count'] = df[financial_cols].isnull().sum(axis=1)
df['_fiscal_sort'] = pd.to_numeric(df['fiscal_year'], errors='coerce')

df = (
    df.sort_values(['ticker', 'missing_count', '_fiscal_sort'], ascending=[True, True, False])
      .drop_duplicates(subset=['ticker'], keep='first')
      .drop(columns=['missing_count', '_fiscal_sort'])
      .reset_index(drop=True)
)

# 5) STRICT REALISM: drop any row with missing financial values (no median imputation)
rows_before_strict = len(df)
df = df.dropna(subset=financial_cols, how='any').copy()
rows_dropped_strict = rows_before_strict - len(df)

# Optional text cleanup
df['extraction_error'] = df['extraction_error'].fillna('')

print('Cleaned shape:', df.shape)
print('Rows removed (metadata_only):', removed_metadata_only)
print('Rows removed (>50% missing financial):', removed_high_missing)
print('Rows removed (strict financial completeness):', rows_dropped_strict)
print('Total rows removed:', rows_start - len(df))

Cleaned shape: (356, 21)
Rows removed (metadata_only): 33
Rows removed (>50% missing financial): 1
Rows removed (strict financial completeness): 5
Total rows removed: 60


In [39]:
# Save cleaned dataset and show before vs after summary
output_path = 'insolvency_clean.csv'

df.to_csv(output_path, index=False)

summary = pd.DataFrame({
    'metric': [
        'rows',
        'columns',
        'duplicate_ticker_rows',
        'financial_missing_values',
        'unique_tickers'
    ],
    'before': [
        len(df_raw),
        len(df_raw.columns),
        int(df_raw.duplicated(subset=['ticker'], keep=False).sum()) if 'ticker' in df_raw.columns else np.nan,
        int(df_raw[financial_cols].isnull().sum().sum()),
        int(df_raw['ticker'].nunique()) if 'ticker' in df_raw.columns else np.nan
    ],
    'after': [
        len(df),
        len(df.columns),
        int(df.duplicated(subset=['ticker'], keep=False).sum()),
        int(df[financial_cols].isnull().sum().sum()),
        int(df['ticker'].nunique())
    ]
})

print(summary)
print(f"\nSaved cleaned file: {output_path}")

                     metric  before  after
0                      rows     416    356
1                   columns      21     21
2     duplicate_ticker_rows      40      0
3  financial_missing_values     343      0
4            unique_tickers     395    356

Saved cleaned file: insolvency_clean.csv
